In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
ONLY    = ''    # blank = every table in the config; or 'KNB1, ADR6'
WORKERS = 4     # tables checked at the same time

In [ ]:
%run ./NB_Config_PROC_B1

In [ ]:
# ── Key check (read-only): does every configured table exist in its source, with its key columns, unique on its key? ──
#   Run before the first Bronze2 load and after adding tables. Nothing is written. DUPLICATES means the configured key
#   is not the table's key: Bronze would keep only one row per key, so fix primary_keys first.
#   Also lists the swap tables that have no <table>_stg folder yet (Dab has not written staging).
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from concurrent.futures import ThreadPoolExecutor

spark.conf.set('spark.sql.parquet.datetimeRebaseModeInRead', 'CORRECTED')
_vl = notebookutils.variableLibrary.getLibrary(VL_NAME)


def vl_value(key):
    try:
        v = _vl[key]
    except Exception:
        return None
    return v.strip().rstrip('/') if isinstance(v, str) and v.strip() else None


groups = {n: g for n, g in globals().items() if isinstance(g, dict) and n.endswith(('_TABLE_CONFIGS', '_DESC_CONFIGS'))}
tables = [(n, cfg) for g in groups.values() for n, cfg in g.items()]
wanted = [w.strip().upper() for w in ONLY.split(',') if w.strip()]
tables = [(n, c) for n, c in tables if not wanted or n.upper() in wanted]


def check(name, cfg):
    key_name = cfg.get('source_lh_key', SOURCE_LH_KEY)
    lh = vl_value(key_name)
    if not lh:
        return name, 'NO VL KEY', f'{VL_NAME}.{key_name} is empty or missing'
    schema = cfg['source_schema'] if 'source_schema' in cfg else SOURCE_SCHEMA
    path = f"{lh}/{schema}/{cfg.get('source_table', name)}" if schema else f"{lh}/{cfg.get('source_table', name)}"
    try:
        if not DeltaTable.isDeltaTable(spark, path):
            return name, 'NOT IN SOURCE', path
        df = spark.read.format('delta').load(path)
    except Exception as e:
        return name, 'NOT IN SOURCE', f'{path} ({str(e)[:80]})'
    cols = {c.lower(): c for c in df.columns}
    keys = list(cfg['primary_keys']) + (['SPRAS'] if cfg.get('spras') and 'spras' in cols and 'SPRAS' not in cfg['primary_keys'] else [])
    missing = [k for k in keys if k.lower() not in cols]
    if missing:
        return name, 'MISSING COLUMN', ', '.join(missing)
    keys = [cols[k.lower()] for k in keys]
    r = df.agg(F.count(F.lit(1)).alias('n'), F.countDistinct(*keys).alias('k')).collect()[0]
    if r['n'] == 0:
        return name, 'EMPTY', path
    if r['n'] != r['k']:
        return name, 'DUPLICATES', f"{r['n'] - r['k']:,} extra rows on key {keys} ({r['n']:,} rows)"
    return name, 'OK', f"{r['n']:,} rows, key {keys}"


with ThreadPoolExecutor(WORKERS) as pool:
    results = list(pool.map(lambda t: check(*t), tables))

line = '═' * 100
print(line)
print(f'Key check: {PIPELINE}, {len(results)} tables, source {SOURCE_LH_KEY} {SOURCE_SCHEMA}')
print(line)
order = ['DUPLICATES', 'MISSING COLUMN', 'NO VL KEY', 'NOT IN SOURCE', 'EMPTY', 'OK']
for status in order:
    rows = sorted(r for r in results if r[1] == status)
    if rows:
        print(f'\n{status} ({len(rows)})')
        for name, _, detail in rows:
            print(f'  {name:<22} {detail}')

# swap tables without staging
for domain, cfg in globals().get('SWAP_DOMAINS', {}).items():
    lh = vl_value(cfg.get('lh_key', SOURCE_LH_KEY))
    folder = cfg.get('tables_folder', DEFAULTS.get('tables_folder', ''))
    stg = cfg.get('stg_suffix', DEFAULTS.get('stg_suffix', '_stg'))
    names = cfg['tables'] if isinstance(cfg['tables'], list) else list(cfg['tables'])
    no_stg = [t for t in names if t != '*' and lh and not notebookutils.fs.exists(f"{lh}/{folder}/{t}{stg}" if folder else f"{lh}/{t}{stg}")]
    print(f'\nSwap {domain}: {len(names) - len(no_stg)} of {len(names)} tables have a {stg} folder'
          + (f'; none yet for: {", ".join(no_stg)}' if no_stg else ''))
print(line)
bad = [r for r in results if r[1] in ('DUPLICATES', 'MISSING COLUMN', 'NO VL KEY')]
print('Result: ' + ('fix the tables above before the first Bronze2 load' if bad else 'keys OK') +
      (f'; {sum(r[1] == "NOT IN SOURCE" for r in results)} table(s) not in the source (Bronze2 skips them)' if any(r[1] == 'NOT IN SOURCE' for r in results) else ''))

In [ ]:
# ── End of run: stop the session to release capacity (15 Nov 2026) ──────────────────────────────────────
# Same rules as end_session() in NB_Utils_Common, which this notebook does not load: skipped in a pipeline run
# (Fabric stops the session itself) and in a reference run; set END_SESSION = False in a cell above to keep it.
if globals().get('END_SESSION', True):
    try:
        _ctx = notebookutils.runtime.context
        if not (_ctx.get('isForPipeline') or _ctx.get('isReferenceRun')):
            print('Session stopped to release capacity (END_SESSION = False keeps it)')
            notebookutils.session.stop()
    except Exception as e:
        if 'STOP_SESSION_REQUEST' in str(e):
            raise    # Fabric stops the session through this marker exception: it must reach the runtime (18 Nov 2026)
        print(f'Session not stopped ({str(e)[:120]})')